In [1]:
import sys
from pathlib import Path


ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
print(f"Root directory is {ROOT_DIR}")

Root directory is /Users/dcanevarollo/Desktop/Projetos/Unesp/ia/bio-variant-prediction


In [2]:
from bio_var_pred.data.download import download_clinvar
from bio_var_pred.data.annotate import annotate_with_snpeff, index
from bio_var_pred.data.compress import bgzip
from bio_var_pred.utils.paths import INTERIM_DATA_DIR


clinvar_vcf = download_clinvar()

clinvar_vcf = annotate_with_snpeff(
    input_vcf=clinvar_vcf,
    output_vcf=(INTERIM_DATA_DIR / "clinvar_annotated.vcf")
)
clinvar_vcf = bgzip(clinvar_vcf)
index(clinvar_vcf)

print(f"ClinVar data stored in {clinvar_vcf}")

clinvar.vcf.gz already exists. Skipping download.
clinvar_annotated.vcf already annotated. Skipping new annotation.
clinvar_annotated.vcf.gz already compressed. Skipping new compress.
clinvar_annotated.vcf.gz already indexed. Skipping new index.
ClinVar data stored in /Users/dcanevarollo/Desktop/Projetos/unesp/ia/bio-variant-prediction/data/interim/clinvar_annotated.vcf.gz


In [3]:
from bio_var_pred.data.parse import parse_clinvar


clinvar_df = parse_clinvar(clinvar_vcf)
clinvar_df.head(10)

Parsing ClinVar: 4434137it [00:20, 218095.19it/s]


,id,chrom,pos_genomic,ref,alt,gene,pos_protein,aa_wt,aa_mut,label
0,1164676,1,930165,G,A,SAMD11,207,Arg,Gln,0
1,1170208,1,930204,G,A,SAMD11,220,Arg,Gln,0
2,1165489,1,930285,G,A,SAMD11,247,Arg,Gln,0
3,1170010,1,930314,C,T,SAMD11,257,His,Tyr,0
4,1167937,1,935779,G,A,SAMD11,284,Gly,Ser,0
5,1167351,1,939429,G,C,SAMD11,51,Trp,Cys,0
6,1166513,1,942451,T,C,SAMD11,169,Trp,Arg,0
7,1166588,1,942481,C,T,SAMD11,221,Pro,Leu,0
8,1167193,1,942577,C,T,SAMD11,307,Thr,Ile,0
9,1164675,1,942649,C,T,SAMD11,331,Thr,Met,0


In [4]:
from bio_var_pred.data.download import download_gnomad


gnomad_vcf = download_gnomad()
gnomad_vcf = INTERIM_DATA_DIR / gnomad_vcf.name
index(gnomad_vcf)

print(f"gnomAD source downloaded to {gnomad_vcf}")

gnomad.exomes.chr1.vcf.bgz already exists. Skipping download.
gnomad.exomes.chr1.vcf.bgz already indexed. Skipping new index.
gnomAD source downloaded to /Users/dcanevarollo/Desktop/Projetos/unesp/ia/bio-variant-prediction/data/interim/gnomad.exomes.chr1.vcf.bgz


In [5]:
from bio_var_pred.data.parse import parse_gnomad


gnomad_df = parse_gnomad(gnomad_vcf, chrom="chr1")
gnomad_df.head(10)

Parsing gnomAD: 17671166it [11:12, 26288.91it/s]


,chrom,pos_genomic,ref,alt,filter,qual,af,ac,an
0,1,11994,T,C,AC0;AS_VQSR,None,NaN,0,0
1,1,12016,G,A,AC0;AS_VQSR,None,NaN,0,0
2,1,12074,T,C,AC0;AS_VQSR,None,0.000000,0,4
3,1,12102,G,A,AC0;AS_VQSR,None,0.000000,0,32
4,1,12106,T,G,AC0;AS_VQSR,None,0.000000,0,34
5,1,12138,C,A,AS_VQSR,None,0.009091,1,110
6,1,12158,C,T,AC0;AS_VQSR,None,0.000000,0,480
7,1,12165,G,A,AC0;AS_VQSR,None,0.000000,0,698
8,1,12168,T,G,AC0;AS_VQSR,None,0.000000,0,626
9,1,12169,A,C,AC0;AS_VQSR,None,0.000000,0,566


In [6]:
df = clinvar_df.merge(
    gnomad_df,
    on=["chrom", "pos_genomic", "ref", "alt"],
    how="left"
)

In [7]:
df["af"] = df["af"].fillna(0.0)

In [8]:
import numpy as np


df["log_af"] = np.log10(df["af"] + 1e-8)

In [9]:
df.shape

(80498, 16)

In [10]:
df.columns

Index(['id', 'chrom', 'pos_genomic', 'ref', 'alt', 'gene', 'pos_protein',
       'aa_wt', 'aa_mut', 'label', 'filter', 'qual', 'af', 'ac', 'an',
       'log_af'],
      dtype='str')

In [11]:
df.head()

,id,chrom,pos_genomic,ref,alt,gene,pos_protein,aa_wt,aa_mut,label,filter,qual,af,ac,an,log_af
0,1164676,1,930165,G,A,SAMD11,207,Arg,Gln,0,NaN,None,0.000501,704.0,1404316.0,-3.299883
1,1170208,1,930204,G,A,SAMD11,220,Arg,Gln,0,NaN,None,0.000944,1341.0,1419824.0,-3.024801
2,1165489,1,930285,G,A,SAMD11,247,Arg,Gln,0,NaN,None,0.000447,651.0,1456402.0,-3.349691
3,1170010,1,930314,C,T,SAMD11,257,His,Tyr,0,NaN,None,0.010014,14562.0,1454086.0,-1.999370
4,1167937,1,935779,G,A,SAMD11,284,Gly,Ser,0,NaN,None,0.001400,2045.0,1461020.0,-2.853959


In [12]:
from bio_var_pred.utils.paths import PROCESSED_DATA_DIR

df["gene"] = df["gene"].astype("category")
df.to_parquet(PROCESSED_DATA_DIR / "variants.parquet")